#### 연습
- data 폴더 안에 가전 폴더의 모든 json파일을 하나의 데이터프레임으로 단순 행 결합
- 데이터에서 결측치를 확인
    - GeneralPolarity 컬럼에서 결측치 발견
- 결측치가 포함된 데이터를 따로 저장(na_df)
- 결측치를 제거
- 'RawText', 'GeneralPolarity' 컬럼을 제외한 나머지 컬럼 제외
- 'GeneralPolarity' 컬럼의 이름을 labels 변경
- 'RawText'는 텍스트 정규화(특수문자 제거, 2칸 이상의 공백 제외, 문자열 앞 뒤 공백 제거)
- labels 데이터에서 -1과 0은 0으로 1은 1로 데이터를 변경 -> 해당 컬럼의 dtype을 int 변경 -> BERTmodel에서 선형 모델로 확률을 예측하기 때문에 labels가 위치 값
- train, test의 형태로 데이터를 9:1의 비율로 나눠준다.
    - labels를 기준으로 계층화 분할
- 데이터프레임을 Dataset의 형태로 변환
- token화 작업은 AutoTokenizer를 이용하여 모델의 이름은 skt/kobert-base-v1을 이용하여 토큰화
- 같은 모델을 로드하여 BertModel + Linear 모델 정의
- Trainer, TrainingArguments를 이용하여 학습 관리 객체 생성
- 학습 -> na_df에서 상위 5개의 RawText을 확인하여 예측

In [247]:
import re
import os
from glob import glob
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments
import numpy as np

In [248]:
# os 라이브러리를 이용하여 파일의 목록 확인
file_path = '../data/가전/'
file_list = os.listdir(file_path)
pd.read_json(file_path + file_list[0])

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects
0,112038,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,SNS,가전,영상/음향가전,(1+1세트) TJ 태진 블루투스 마이크 / 무선 노래방 마이크,3,323,73,20221110,0.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '우리나라 ..."
1,114572,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,SNS,가전,영상/음향가전,(1등급)삼성 QLED 4K TV 138cm(55형) KQ55QT67AFXKR+삼성...,1,317,70,20221119,-1.0,"[{'Aspect': '품질', 'SentimentText': '겉 부분에 여기저기..."
2,114573,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,SNS,가전,영상/음향가전,4K HDMI 2.0 양방향 선택기,1,311,70,20221113,0.0,"[{'Aspect': '기능', 'SentimentText': ' 그래도 괜찮은 건..."
3,114574,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,306,84,20221121,-1.0,"[{'Aspect': '품질', 'SentimentText': '너무 별로예요 ㅡㅡ..."
4,114575,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,303,73,20221121,-1.0,"[{'Aspect': '소음', 'SentimentText': '소음이 섞여서 나는..."
...,...,...,...,...,...,...,...,...,...,...,...,...
96,114667,이전부터 구매하고 싶어서 계속 눈여겨보다 구매한 락클래식이에요. 포장을 제거하고 제...,SNS,가전,영상/음향가전,엠지텍 락클래식Q9900 (정품),1,290,61,20221120,-1.0,"[{'Aspect': '품질', 'SentimentText': '마감은 좀 문제가 ..."
97,114668,요즘 집에서 작업하면서 핸드폰으로 음악을 들으니 전화를 하거나 핸드폰을 이용할때 자...,SNS,가전,영상/음향가전,오아 아이브릭 휴대용 블루투스 미니 스피커,1,338,78,20221110,-1.0,"[{'Aspect': '디자인', 'SentimentText': '디자인이 좀 그렇..."
98,114669,"처음 들어보는 생소한 브랜드의 tv라 걱정하면서 구입했는데, 역시나 후회 중입니다....",SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,333,79,20221116,-1.0,"[{'Aspect': '소음', 'SentimentText': '별로 소리를 키우지..."
99,114670,최신 기종이라고 해서 기대했는데 구기종보다 못하네요. 소재가 별로여서 예쁜 디자인이...,SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,324,74,20221116,-1.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '최신 기종..."


In [249]:
# glob 라이브러리의 glob을 이용
file_list2 = glob('../data/가전/*.json')
pd.read_json(file_list2[0])

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects
0,112038,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,SNS,가전,영상/음향가전,(1+1세트) TJ 태진 블루투스 마이크 / 무선 노래방 마이크,3,323,73,20221110,0.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '우리나라 ..."
1,114572,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,SNS,가전,영상/음향가전,(1등급)삼성 QLED 4K TV 138cm(55형) KQ55QT67AFXKR+삼성...,1,317,70,20221119,-1.0,"[{'Aspect': '품질', 'SentimentText': '겉 부분에 여기저기..."
2,114573,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,SNS,가전,영상/음향가전,4K HDMI 2.0 양방향 선택기,1,311,70,20221113,0.0,"[{'Aspect': '기능', 'SentimentText': ' 그래도 괜찮은 건..."
3,114574,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,306,84,20221121,-1.0,"[{'Aspect': '품질', 'SentimentText': '너무 별로예요 ㅡㅡ..."
4,114575,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,303,73,20221121,-1.0,"[{'Aspect': '소음', 'SentimentText': '소음이 섞여서 나는..."
...,...,...,...,...,...,...,...,...,...,...,...,...
96,114667,이전부터 구매하고 싶어서 계속 눈여겨보다 구매한 락클래식이에요. 포장을 제거하고 제...,SNS,가전,영상/음향가전,엠지텍 락클래식Q9900 (정품),1,290,61,20221120,-1.0,"[{'Aspect': '품질', 'SentimentText': '마감은 좀 문제가 ..."
97,114668,요즘 집에서 작업하면서 핸드폰으로 음악을 들으니 전화를 하거나 핸드폰을 이용할때 자...,SNS,가전,영상/음향가전,오아 아이브릭 휴대용 블루투스 미니 스피커,1,338,78,20221110,-1.0,"[{'Aspect': '디자인', 'SentimentText': '디자인이 좀 그렇..."
98,114669,"처음 들어보는 생소한 브랜드의 tv라 걱정하면서 구입했는데, 역시나 후회 중입니다....",SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,333,79,20221116,-1.0,"[{'Aspect': '소음', 'SentimentText': '별로 소리를 키우지..."
99,114670,최신 기종이라고 해서 기대했는데 구기종보다 못하네요. 소재가 별로여서 예쁜 디자인이...,SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,324,74,20221116,-1.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '최신 기종..."


In [250]:
# 모든 데이터프레임을 로드하여 하나의 데이터프레임으로 생성
# 누적으로 데이터프레임이 결합되는 공간 -> 빈 데이터프레임
total_df = pd.DataFrame()
# file_list2만큼 반복 실행하는 반복문을 생성
for file_path in file_list2:
    df = pd.read_json(file_path)
    total_df = pd.concat([total_df, df], axis = 0)
total_df.reset_index(drop=True, inplace=True)
total_df

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects
0,112038,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,SNS,가전,영상/음향가전,(1+1세트) TJ 태진 블루투스 마이크 / 무선 노래방 마이크,3,323,73,20221110,0.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '우리나라 ..."
1,114572,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,SNS,가전,영상/음향가전,(1등급)삼성 QLED 4K TV 138cm(55형) KQ55QT67AFXKR+삼성...,1,317,70,20221119,-1.0,"[{'Aspect': '품질', 'SentimentText': '겉 부분에 여기저기..."
2,114573,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,SNS,가전,영상/음향가전,4K HDMI 2.0 양방향 선택기,1,311,70,20221113,0.0,"[{'Aspect': '기능', 'SentimentText': ' 그래도 괜찮은 건..."
3,114574,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,306,84,20221121,-1.0,"[{'Aspect': '품질', 'SentimentText': '너무 별로예요 ㅡㅡ..."
4,114575,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,303,73,20221121,-1.0,"[{'Aspect': '소음', 'SentimentText': '소음이 섞여서 나는..."
...,...,...,...,...,...,...,...,...,...,...,...,...
4051,112332,사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...,SNS,가전,계절가전,LG 듀얼히트 2in1 에어컨 FQ20HADWB2 (65.9㎡＋22.8㎡) [전국기...,2,304,68,20221108,1.0,"[{'Aspect': '디자인', 'SentimentText': '투박하기도하고 디..."
4052,112333,이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...,SNS,가전,계절가전,LG 멀티형 에어컨 FQ17V8WWF2 기본설치무료 서울경기(지역별배송비확인),4,329,80,20221108,1.0,"[{'Aspect': '조작성', 'SentimentText': '인공지능 조작이 ..."
4053,112334,"이 제품을 추천하는 이유는요! 일단 LED를 통해 공기질을 눈으로 확인할 수 있고,...",SNS,가전,계절가전,LG 미니 공기청정기 골드 AP111MGHA.AKOR,5,317,80,20221116,1.0,"[{'Aspect': '기능', 'SentimentText': ' LED를 통해 공..."
4054,112335,"인터넷에서 구매하였는데요, 이 공기청정기 처음 발견하고, 처음에는 오잉? 이게뭐지?...",SNS,가전,계절가전,LG 미니 공기청정기 골드 AP111MGHA.AKOR,4,316,70,20221116,1.0,"[{'Aspect': '편의성', 'SentimentText': ' 딱 컵홀더처럼 ..."


In [251]:
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4056 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 380.4+ KB


In [252]:
pd.concat([pd.read_json(file_name) for file_name in file_list2]).info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 411.9+ KB


In [253]:
# python 기본 내장된 map() 함수 이용 -> map(function, list)
pd.concat( list(map(lambda file_name : pd.read_json(file_name), file_list2)) ).info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 411.9+ KB


In [254]:
# 결측치의 개수 확인
total_df['GeneralPolarity'].isna().sum()

np.int64(378)

In [255]:
# 결측치인 데이터들을 다른 변수에 저장
na_df = total_df.loc[total_df['GeneralPolarity'].isna(), ]
na_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 378 entries, 13 to 3944
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            378 non-null    int64  
 1   RawText          378 non-null    object 
 2   Source           378 non-null    object 
 3   Domain           378 non-null    object 
 4   MainCategory     378 non-null    object 
 5   ProductName      378 non-null    object 
 6   ReviewScore      378 non-null    int64  
 7   Syllable         378 non-null    int64  
 8   Word             378 non-null    int64  
 9   RDate            378 non-null    int64  
 10  GeneralPolarity  0 non-null      float64
 11  Aspects          378 non-null    object 
dtypes: float64(1), int64(5), object(6)
memory usage: 38.4+ KB


In [256]:
# total_df = total_df.loc[~total_df['GeneralPolarity'].isna()]

In [257]:
# 결측치를 제거 total_df에서
total_df.dropna(inplace=True)

In [258]:
# 특정 컬럼 두개를 제외한 다른 컬럼은 제외 -> 특정 컬럼 2개만 선택
df = total_df[['RawText', 'GeneralPolarity']]

In [259]:
# 특정 컬럼의 이름을 변경 rename()
df = df.rename(columns={'GeneralPolarity' : 'labels'})
df

,RawText,labels
0,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,0.0
1,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,-1.0
2,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,0.0
3,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,-1.0
4,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,-1.0
...,...,...
4051,사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...,1.0
4052,이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...,1.0
4053,"이 제품을 추천하는 이유는요! 일단 LED를 통해 공기질을 눈으로 확인할 수 있고,...",1.0
4054,"인터넷에서 구매하였는데요, 이 공기청정기 처음 발견하고, 처음에는 오잉? 이게뭐지?...",1.0


In [ ]:
# def normalize(text : str) -> str:
#     text = re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text

In [ ]:
# df['RawText'].map(normalize)

0       엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....
1       누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요 일단 겉 부분에 ...
2       노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...
3       너무 별로예요 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요 음질이 거의 입 안...
4       소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...
                              ...                        
4051    사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...
4052    이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...
4053    이 제품을 추천하는 이유는요 일단 LED를 통해 공기질을 눈으로 확인할 수 있고 터...
4054    인터넷에서 구매하였는데요 이 공기청정기 처음 발견하고 처음에는 오잉 이게뭐지 했어요...
4055    제품 소개 골드 빛이 도는 작은 제품으로 색감도 마음에 들고 사이즈도 참 아담하니 ...
Name: RawText, Length: 3678, dtype: object

In [262]:
df['labels'].value_counts()

labels
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [263]:
# RawText는 문자의 정규화
# labels는 -1인 데이터는 0으로 타입을 int로 변경

# labels 컬럼의 데이터의 타입을 int로 고정시킨다
df['labels'] = df['labels'].astype(int)

In [ ]:
def normalize(x):
    # x -> apply() 함수를 이용하여 들어오는 Series 형태의 데이터
    x['RawText'] = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', " ", x['RawText'])
    x['RawText'] = re.sub(r'\s+', ' ', x['RawText'])
    # if x['labels'] == -1:
    #     x['labels'] = 0
    # -1, 0, 1의 labels는 예측 확률에서 -1의 위치가 존재하지 않음으로 에러가 발생
    # labels 데이터에 1을 증가
    x['labels'] += 1
    return x


df = df.apply(normalize, axis=1)
# 컬럼을 기준으로 데이터를 나눠서 보여준다
# axis = 0
# df['RawText']를 x에 대입하여 한번 실행
# df['labels']를 x에 대입하여 한번 실행

# axis = 1
# df.iloc[0, ] x대입
# df.iloc[1. ] x대입 ...

In [265]:
# df['labels'] = df['labels'].map(lambda x: 1 if x == 1.0 else 0)
# df['labels'].info()

In [280]:
df['labels'].value_counts()

labels
2    2220
1     944
0     514
Name: count, dtype: int64

In [281]:
# train, test 데이터셋으로 분할 -> labels의 데이터로 계층화
train_df, test_df = train_test_split(
    df, test_size=0.1, random_state=42, stratify=df['labels']
)
len(train_df)

3310

In [282]:
# DataFrame -> Dataset 형태로 파싱
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

In [283]:
train_ds

Dataset({
    features: ['RawText', 'labels'],
    num_rows: 3310
})

In [284]:
# Tokenizer
MODEL_NAME = 'skt/kobert-base-v1'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

In [285]:
def tok_fn(batch):
    # batch -> Dataset에서 map() 함수의 기본값이 데이터를 특정 개수로 묶어서 보낸다
    result = tokenizer(batch['RawText'], truncation=True, max_length=128)
    # 토큰의 개수의 제한을 두고 해당 토큰보다 길다면 잘라준다
    return result

In [286]:
train_tok = train_ds.map(tok_fn, batched=True, remove_columns=['RawText'])
test_tok = test_ds.map(tok_fn, batched=True, remove_columns=['RawText'])

Map: 100%|██████████| 368/368 [00:00<00:00, 913.76 examples/s]


In [287]:
# BertModel + Linear 모델의 정의
class BERTClsHead(nn.Module):
    def __init__(self, model_name, num_label=2, dropout=0.1):
        super().__init__()
        # 기존의 학습이 된 모델 로드
        self.backbone = BertModel.from_pretrained(model_name)
        # 로드한 모델에서 출력 차원의 개수를 저장
        hidden = self.backbone.config.hidden_size

        # 일정 부분 소실시키는 Dropout
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_label)

        # 작업의 안정성을 위해서 tokenizer에서 사용하는 패딩 토큰의 id를 모델에 패딩 id 대입
        self.backbone.config.pad_token_id = tokenizer.pad_token_id

    # 순전파 함수
    def forward(self, input_ids = None, attention_mask = None, labels = None, **kwargs):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        pooled = out.last_hidden_state[:, 0]

        logits = self.classifier(self.dropout(pooled))

        result = {'Logits' : logits}

        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            result['loss'] = loss

        return result

In [295]:
# 모델 생성
model = BERTClsHead(MODEL_NAME, num_label=3)

In [303]:
# 평가에서 사용할 함수 정의
def metrics(eval_pred):
    logits, y = eval_pred
    pred = logits.argmax(-1)
    return {
        'accuracy_score' : accuracy_score(y, pred),
        'f1_score' : f1_score(y, pred, average = 'macro')
    }

In [304]:
# Trainer가 사용할 설정 계수를 정의
args = TrainingArguments(
    output_dir = './kobert_from_bertmodel',
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    num_train_epochs = 2,     # 실제 횟수 -> 일반적으로 3~5회
    learning_rate = 5e-5,     # 5e-5, 4e-5, 3e-5
    weight_decay = 0.01,
    warmup_ratio = 0.01,
    logging_steps = 50,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1',
    greater_is_better = True,
    report_to = []
)

In [305]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = train_tok,
    eval_dataset = test_tok,
    tokenizer = tokenizer,
    compute_metrics = metrics
)

C:\Users\student\AppData\Local\Temp\ipykernel_6876\3400107868.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [306]:
eval_res = trainer.evaluate()
print('평가의 결과 :', eval_res)

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


평가의 결과 : {'eval_loss': 1.108737587928772, 'eval_model_preparation_time': 0.0044, 'eval_accuracy_score': 0.2717391304347826, 'eval_f1_score': 0.2352876357154432, 'eval_runtime': 47.7154, 'eval_samples_per_second': 7.712, 'eval_steps_per_second': 0.964}


In [307]:
# 새로운 문장에 대해서 평가
samples = na_df['RawText'].head().to_list()
samples

['귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만 재질만 바꾸어도 안 빠질것 같은데 000 것은 귀에 꼽으면 안 빠져서 사용할 때 아무 문제 없고 멍멍한 현상도 없는데 아무튼 만드는 이가 000 것과 사용 비교 해보고 하다 못해 재질이라도 바꾸든지 반품도 못하고 통화 중 몇 번씩 빠져서 사용 불가능 합니다.통화음은 상대방쪽 발음이 잘 안들린다 하고 음악 들을때는 스테레오 됩니다.통화시 사용하는 사람에게는 음질이 안 좋아 비추천내쪽에서 말하는것이 상대가 잘 발음이 안들린다 해서 중간에 통화를 중단하는 경우가 여러번 대기업 회사에서 시착도 안해보고 완성된 물건이라 판매 하는지그냥 만드는건지 ...오픈해서 반품도 못하고 너무하네요.',
 '아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거워서 떨어지거나 하면 아이가 다칠까 봐 걱정이 되었거든요. 그런데 이 TV는 마감 퀄리티가 별로입니다. 아이가 자칫하다 TV를 만지면 손이 베일까 봐 걱정이에요. 저희 남편은 연결을 하다가 손이 살짝 까졌습니다.전에 쓰던 TV보다 무게가 가벼워서, 아이가 혹여나 다칠 일이 줄어들지 않을까 구입한 TV인데 마감이 별로라 더 걱정이네요. 마감이 깔끔하지 못한 부분은 사포로 살짝 갈아볼까 하는데 그렇게 되면 디자인도 떨어지게 될까 봐 노심초사하는 중입니다. 처음부터 마감이 괜찮았으면 이런 걱정도 없었을 텐데 여러모로 아쉽네요.',
 '화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나 하지 않았는데 사용한 지 한 달이 되는 지금, 갑자기 화면에 노이즈가 생기네요. 큰맘 먹고 구입한 TV라 혹여나 화면에 흠집이라도 생길까 봐 조심히 청소했습니다. 배신감이 좀 드네요.이 TV의 가장 큰 장점이 화질이라고 생각했던 사람입니다. 하지만 노이즈 때문에 선명도가 떨이 지고 화면이 깨끗하지 않네요. a/s가 잘 될지는 모르겠지만 한 달도 안 된 시점에서 가장 중요한 화질이 떨어진다는 점은 큰 문제점이지 않을까요?a/s가 어렵다고 하

In [308]:
enc = tokenizer(
    samples,
    return_tensors = 'pt',
    padding = True,
    truncation = True,
    max_length = 128
)

In [309]:
with torch.no_grad():
    out = model(**enc)
    probs = torch.softmax(out['Logits'], dim=-1)

In [316]:
for s, p in zip(samples, probs):
    print(f'''
    리뷰 데이터 : {s[:40]}  
    부정 : {p[0]:.3f},  중간 : {p[1]:.3f},  긍정 : {p[2]:.3f}   |  예측 : {p.argmax()}
''')


    리뷰 데이터 : 귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만   
    부정 : 0.299,  중간 : 0.367,  긍정 : 0.334   |  예측 : 1


    리뷰 데이터 : 아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거  
    부정 : 0.331,  중간 : 0.342,  긍정 : 0.327   |  예측 : 1


    리뷰 데이터 : 화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나  
    부정 : 0.318,  중간 : 0.388,  긍정 : 0.294   |  예측 : 1


    리뷰 데이터 : 이번에 이사하면서 우리 따님께서 방에 TV가 있으면 좋겠다고 하여, 방에  
    부정 : 0.325,  중간 : 0.341,  긍정 : 0.334   |  예측 : 1


    리뷰 데이터 : 기존에 사용하던 무선이어폰이 오래되어서 배터리가 광탈하는 바람에 새로운   
    부정 : 0.314,  중간 : 0.374,  긍정 : 0.312   |  예측 : 1

